In [1]:
# ==========================================
# CELDA 1: Extracción de Datos (ETL Base)
# ==========================================
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# 1. Conexión a la Base de Datos
# Ajusta tus credenciales según tu entorno local
user = "Joasro"
password = "Akriila123." 
host = "localhost"
db = "dss_academico_unah"

engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{db}")

print("Extrayendo datos de la universidad...")

# Extraemos las tablas clave
df_historial = pd.read_sql("""
    SELECT h.Hash_Cuenta, h.ID_Clase, h.Estado, h.Periodo_Cursado, 
           m.Unidades_Valorativas, m.Codigo_Oficial, m.Plan_Perteneciente
    FROM Historial_Academico h
    JOIN Malla_Curricular m ON h.ID_Clase = m.ID_Clase
""", engine)

df_estudiantes = pd.read_sql("SELECT Hash_Cuenta, Plan_Estudio, Ano_Ingreso FROM Estudiantes", engine)
df_malla = pd.read_sql("SELECT ID_Clase, Codigo_Oficial, Nombre_Clase, Unidades_Valorativas, Plan_Perteneciente, Prerrequisitos FROM Malla_Curricular", engine)
df_censo = pd.read_sql("SELECT Hash_Cuenta, ID_Clase FROM censo_periodo_actual", engine)

print(f"Datos extraídos: {len(df_historial)} registros históricos, {len(df_estudiantes)} estudiantes activos.")

Extrayendo datos de la universidad...
Datos extraídos: 2613 registros históricos, 70 estudiantes activos.


In [2]:
# ==========================================
# CELDA 2: Funciones de Dominio Académico
# ==========================================

EQUIVALENCIAS_REVERSAS = {
    'IS-110': 'ISC-101', 'MM-314': 'ISC-101', 'IS-210': 'ISC-102', 
    'IS-310': 'ISC-211', 'IS-410': 'ISC-103', 'IS-501': 'ISC-321',
    'IS-601': 'ISC-422', 'IS-602': 'ISC-341', 'IS-702': 'ISC-306', 
    'IS-311': 'IE-326',  'IS-510': 'IE-326',  'IS-511': 'ISC-331',
    'IS-611': 'ISC-332', 'IS-412': 'ISC-333', 'IS-512': 'ISC-334', 
    'IS-115': 'ISC-552', 'IS-802': 'ISC-408'
}

def cumple_prerrequisitos(req_text, aprobadas_set, uv_est):
    """Evalúa si el alumno cumple los requisitos para llevar una clase."""
    if not req_text or str(req_text).lower() in ['ninguno', 'nan', 'null', '']:
        return True
    if "140" in str(req_text):
        return uv_est >= 140
    reqs = [r.strip().upper() for r in str(req_text).split(',')]
    return all(r in aprobadas_set for r in reqs)

def extender_aprobadas(aprobadas_base):
    """Si aprobó una clase vieja, se asume que aprobó su equivalente nueva."""
    extendidas = set(aprobadas_base)
    for vieja, nueva in EQUIVALENCIAS_REVERSAS.items():
        if vieja in extendidas:
            extendidas.add(nueva)
    return extendidas

In [3]:
# ==========================================
# CELDA 3: Construcción del Set de Entrenamiento Histórico (Filtrado por Carrera)
# ==========================================
print("Reconstruyendo el pasado para entrenar a la IA (Solo clases de Sistemas)...")

training_data = []

# Ordenamos los periodos cronológicamente
periodos = sorted(df_historial['Periodo_Cursado'].unique(), 
                  key=lambda x: (int(x.split('-')[1]), int(x.split('-')[0])))

dict_estudiantes = df_estudiantes.set_index('Hash_Cuenta').to_dict('index')

for i in range(1, len(periodos)):
    periodo_actual = periodos[i]
    historia_previa = df_historial[df_historial['Periodo_Cursado'].apply(lambda x: periodos.index(x) < i)]
    matriculas_actuales = df_historial[df_historial['Periodo_Cursado'] == periodo_actual]

    for hash_est, info in dict_estudiantes.items():
        plan = info['Plan_Estudio']
        hist_estudiante = historia_previa[historia_previa['Hash_Cuenta'] == hash_est]
        if hist_estudiante.empty: continue 
            
        aprobadas_antes = set(hist_estudiante[hist_estudiante['Estado'] == 'Aprobado']['Codigo_Oficial'])
        aprobadas_antes = extender_aprobadas(aprobadas_antes)
        uv_antes = hist_estudiante[hist_estudiante['Estado'] == 'Aprobado']['Unidades_Valorativas'].sum()
        
        clases_matriculadas = set(matriculas_actuales[matriculas_actuales['Hash_Cuenta'] == hash_est]['Codigo_Oficial'])
        
        # 🔥 EL FILTRO MÁGICO: Solo tomamos clases del plan que empiecen con IS, ISC o IE
        clases_plan = df_malla[
            (df_malla['Plan_Perteneciente'] == plan) & 
            (df_malla['Codigo_Oficial'].str.startswith(('IS', 'ISC', 'IE')))
        ]
        
        for _, clase in clases_plan.iterrows():
            cod = clase['Codigo_Oficial']
            if cod in aprobadas_antes: continue 
                
            if cumple_prerrequisitos(clase['Prerrequisitos'], aprobadas_antes, uv_antes):
                training_data.append({
                    'UV_Acumuladas': uv_antes,
                    'Es_Egresando': 1 if uv_antes >= 140 else 0,
                    'Plan_n': 1 if plan == '2025' else 0,
                    'Matriculo_Real': 1 if cod in clases_matriculadas else 0 
                })

df_train = pd.DataFrame(training_data)
print(f"✅ Set de entrenamiento creado: {len(df_train)} escenarios históricos analizados.")
display(df_train['Matriculo_Real'].value_counts())

Reconstruyendo el pasado para entrenar a la IA (Solo clases de Sistemas)...
✅ Set de entrenamiento creado: 1551 escenarios históricos analizados.


Matriculo_Real
1    1147
0     404
Name: count, dtype: int64

In [4]:
# ==========================================
# CELDA 4: Entrenamiento del Motor Predictivo
# ==========================================
from sklearn.model_selection import train_test_split

X = df_train[['UV_Acumuladas', 'Es_Egresando', 'Plan_n']]
y = df_train['Matriculo_Real']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo_base = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    random_state=42,
    eval_metric='logloss'
)

print("Entrenando red neuronal XGBoost...")
modelo_base.fit(X_train, y_train)

y_pred = modelo_base.predict(X_test)
print(f"Precisión del modelo base (Solo con historial): {accuracy_score(y_test, y_pred):.2f}")

Entrenando red neuronal XGBoost...
Precisión del modelo base (Solo con historial): 0.83


In [5]:
# ==========================================
# CELDA 5: Predicción IPAC-2026 + Efecto Censo (Filtrado por Carrera)
# ==========================================
print("Calculando demanda final para el próximo periodo (Solo clases de Sistemas)...")

prediccion_actual = []
aprobadas_totales = df_historial[df_historial['Estado'] == 'Aprobado'].groupby('Hash_Cuenta')['Codigo_Oficial'].apply(set).to_dict()

for hash_est, info in dict_estudiantes.items():
    plan = info['Plan_Estudio']
    aprobadas = extender_aprobadas(aprobadas_totales.get(hash_est, set()))
    
    # Calculamos UV actuales
    uv_actuales = sum([df_malla[df_malla['Codigo_Oficial'] == c]['Unidades_Valorativas'].values[0] 
                       for c in aprobadas if c in df_malla['Codigo_Oficial'].values])
    
    # Filtro por Carrera (IS, ISC, IE)
    clases_plan = df_malla[
        (df_malla['Plan_Perteneciente'] == plan) & 
        (df_malla['Codigo_Oficial'].str.startswith(('IS', 'ISC', 'IE')))
    ]
    
    for _, clase in clases_plan.iterrows():
        cod = clase['Codigo_Oficial']
        id_clase = clase['ID_Clase']
        
        if cod in aprobadas: continue
            
        if cumple_prerrequisitos(clase['Prerrequisitos'], aprobadas, uv_actuales):
            # Verificamos si el alumno (de los 70) llenó el censo
            en_censo = 1 if ((df_censo['Hash_Cuenta'] == hash_est) & (df_censo['ID_Clase'] == id_clase)).any() else 0
            
            prediccion_actual.append({
                'Hash_Cuenta': hash_est,
                'ID_Clase': id_clase,
                'Codigo_Oficial': cod,
                'Nombre_Clase': clase['Nombre_Clase'],
                'UV_Acumuladas': uv_actuales,
                'Es_Egresando': 1 if uv_actuales >= 140 else 0,
                'Plan_n': 1 if plan == '2025' else 0,
                'En_Censo': en_censo
            })

df_ipac2026 = pd.DataFrame(prediccion_actual)

# 1. Predicción Base con XGBoost
X_ipac = df_ipac2026[['UV_Acumuladas', 'Es_Egresando', 'Plan_n']]
df_ipac2026['Probabilidad_ML'] = modelo_base.predict_proba(X_ipac)[:, 1]

# 2. Multiplicador por Censo Actual
df_ipac2026['Probabilidad_Final'] = np.clip(df_ipac2026['Probabilidad_ML'] + (df_ipac2026['En_Censo'] * 0.35), 0, 1)

# --- BLOQUE DE EXPANSIÓN ACTUALIZADO (MUESTRA = 70) ---

# 3. Calculamos la demanda total observada en la muestra
demanda_final = df_ipac2026.groupby(['Codigo_Oficial', 'Nombre_Clase'])['Probabilidad_Final'].sum().reset_index()
demanda_final.rename(columns={'Probabilidad_Final': 'Demanda_Muestra'}, inplace=True)

# 4. APLICAMOS EL FACTOR DE EXPANSIÓN REAL
POBLACION_TOTAL = 222
MUESTRA = 70  # <--- Actualizado a 70
FACTOR_EXPANSION = POBLACION_TOTAL / MUESTRA

demanda_final['Cupos_Estimados'] = demanda_final['Demanda_Muestra'] * FACTOR_EXPANSION
demanda_final['Cupos_Estimados'] = np.ceil(demanda_final['Cupos_Estimados']).astype(int)

# Limpieza y Orden
demanda_final = demanda_final[demanda_final['Cupos_Estimados'] > 0]
demanda_final = demanda_final.sort_values(by='Cupos_Estimados', ascending=False)

# ==========================================
# MOSTRAR RESULTADOS
# ==========================================
pd.set_option('display.max_rows', None)
print("🎯 RESULTADO FINAL: Demanda Proyectada IPAC-2026")
print(f"📊 Muestra Analizada: {MUESTRA} estudiantes ({ (MUESTRA/POBLACION_TOTAL)*100 :.1f}% de la población)")
print(f"📈 Factor de Proyección: {FACTOR_EXPANSION:.2f}")
print("-" * 60)

display(demanda_final)
pd.reset_option('display.max_rows')

Calculando demanda final para el próximo periodo (Solo clases de Sistemas)...
🎯 RESULTADO FINAL: Demanda Proyectada IPAC-2026
📊 Muestra Analizada: 70 estudiantes (31.5% de la población)
📈 Factor de Proyección: 3.17
------------------------------------------------------------


,Codigo_Oficial,Nombre_Clase,Demanda_Muestra,Cupos_Estimados
1,IS-115,Seminario de Investigación,16.208025,52
5,IS-512,Sistemas Operativos II,5.790274,19
12,IS-711,Diseño Digital,5.772874,19
17,IS-820,Finanzas Administrativas,5.790274,19
16,IS-811,Seguridad Informática,5.790274,19
14,IS-721,Contabilidad I,4.825228,16
15,IS-802,Ingeniería del Software,4.825228,16
24,IS-911,Microprocesadores,4.825228,16
26,IS-913,Diseño de Compiladores,4.807828,16
30,ISC-204,Paradigmas de Programación,4.614145,15


In [6]:
# Guardar resultados para el optimizador
demanda_final.to_csv('../data/demanda_proyectada_2026.csv', index=False)
print("✅ Archivo 'demanda_proyectada_2026.csv' guardado en la carpeta /data.")

✅ Archivo 'demanda_proyectada_2026.csv' guardado en la carpeta /data.
